# Nuclear IT Hack — финальное решение

**Есть совместимый BEST → сразу используем его. Нет ни BEST, ни LAST → обучаем Unet++ заново.** Один выходной 8-битный ДОЭ 128×128, четыре восстановления Френеля. Без benchmark, без SSIM fine-tune на 110 эпох, без попиксельной оптимизации при инференсе.

## 1. Конфигурация и зависимости

In [1]:
%pip install -q segmentation-models-pytorch tqdm pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import math, os, sys, random, json, importlib, hashlib, time
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

ROOT = Path.cwd()
DATA_ROOT = ROOT / 'data'
CKPT_DIR = ROOT / 'checkpoints'
OUT = ROOT / 'outputs' / 'final_submission'
CKPT_DIR.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if AMP:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

TRAIN_EPOCHS_IF_NO_CHECKPOINT = 30
TRAIN_QUARTETS_PER_EPOCH = 4096
VAL_QUARTETS = 128
BATCH_SIZE = 2
LR = 2e-4
PATIENCE = 9
MIN_EPOCHS = 14
MAX_TRAIN_HOURS = 2.25

INPUT_FILES = [DATA_ROOT/'contest'/f'input_{i}.png' for i in range(1,5)]
READY_MARKER = ROOT/'CODEX_READY.json'
if READY_MARKER.exists(): READY_MARKER.unlink()

FINAL_BEST = CKPT_DIR/'best_smp_final.pt'
FINAL_LAST = CKPT_DIR/'last_smp_final.pt'

CHECKPOINT_CANDIDATES = [
    CKPT_DIR/'best_smp_competition.pt',
    CKPT_DIR/'best_smp_physics_v7.pt',
    FINAL_BEST,
    CKPT_DIR/'last_smp_physics_v7.pt',
]
print('DEVICE:',DEVICE,'| ROOT:',ROOT)
print('Checkpoint candidates:',*[str(p) for p in CHECKPOINT_CANDIDATES if p.exists()],sep='\n  ')


DEVICE: cuda | ROOT: d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack
Checkpoint candidates:
  d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\checkpoints\best_smp_competition.pt
  d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\checkpoints\best_smp_physics_v7.pt
  d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\checkpoints\last_smp_physics_v7.pt


## 2. Физика Френеля и 8-битный ДОЭ

In [ ]:
TEAM_FRESNEL_SOURCE = 'import math\nimport cmath\nimport torch\n\nDISTANCES = (0.0241, 0.0253, 0.0265, 0.0277)\n\n\ndef fresnel_field(phase: torch.Tensor, distance: float,\n                  wavelength: float = 532e-9,\n                  pixel_pitch: float = 10e-6) -> torch.Tensor:\n    """Single-FFT Fresnel propagation. Input: phase (B, 1, N, N), rad; output: complex field."""\n    if phase.ndim != 4 or phase.shape[1] != 1 or phase.shape[-2] != phase.shape[-1]:\n        raise ValueError("phase must have shape (B, 1, N, N)")\n    if not phase.is_floating_point():\n        raise TypeError("phase must be a floating-point tensor")\n    if distance <= 0:\n        raise ValueError("distance must be positive")\n\n    n = phase.shape[-1]\n    indices = torch.arange(n, device=phase.device, dtype=phase.dtype) - n // 2\n    u, v = torch.meshgrid(indices * pixel_pitch, indices * pixel_pitch, indexing="ij")\n    chirp_in = torch.exp(1j * math.pi / (wavelength * distance) * (u**2 + v**2))\n    field = torch.exp(1j * phase) * chirp_in\n\n    spectrum = torch.fft.fftshift(\n        torch.fft.fft2(torch.fft.ifftshift(field, dim=(-2, -1)), dim=(-2, -1)),\n        dim=(-2, -1),\n    )\n\n    output_pitch = wavelength * distance / (n * pixel_pitch)\n    x, y = torch.meshgrid(indices * output_pitch, indices * output_pitch, indexing="ij")\n    chirp_out = torch.exp(1j * math.pi / (wavelength * distance) * (x**2 + y**2))\n    prefactor = (pixel_pitch**2 / (1j * wavelength * distance)) * cmath.exp(\n        1j * 2 * math.pi * distance / wavelength\n    )\n    return prefactor * chirp_out * spectrum\n\n\ndef fresnel_propagate(phase: torch.Tensor, distance: float) -> torch.Tensor:\n    return fresnel_field(phase, distance).abs()\n\n\ndef reconstruct_four(phase: torch.Tensor) -> torch.Tensor:\n    return torch.cat([fresnel_propagate(phase, z) for z in DISTANCES], dim=1)\n'
phys_path=ROOT/'src'/'physics'/'fresnel.py'
phys_path.parent.mkdir(parents=True,exist_ok=True)
for initializer in (ROOT/'src'/'__init__.py',ROOT/'src'/'physics'/'__init__.py'):
    if not initializer.exists():initializer.write_text('',encoding='utf-8')
if not phys_path.exists():
    phys_path.write_text(TEAM_FRESNEL_SOURCE,encoding='utf-8')
    print('Created EXACT team physics:',phys_path)
print('Team physics SHA256:',hashlib.sha256(phys_path.read_bytes()).hexdigest())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
importlib.invalidate_caches()
from src.physics.fresnel import DISTANCES, fresnel_field, reconstruct_four
assert DISTANCES==(0.0241,0.0253,0.0265,0.0277)

def norm_amp(a):return a/a.amax(dim=(-2,-1),keepdim=True).clamp_min(1e-8)

def phase_to_png(phase,path):
    phase=torch.atan2(phase.detach().float().sin(),phase.detach().float().cos())
    arr=torch.round((phase+math.pi)*255/(2*math.pi)).clamp(0,255).byte().cpu().numpy()
    Image.fromarray(arr,mode='L').save(path)

def phase_from_png(path):
    ar=np.asarray(Image.open(path).convert('L'),dtype=np.float32).copy()
    return (torch.from_numpy(ar).to(DEVICE)*(2*math.pi/255)-math.pi)[None,None]

def q8(phase):
    wrapped=torch.atan2(phase.sin(),phase.cos())
    code=torch.round((wrapped+math.pi)*255/(2*math.pi)).clamp(0,255)
    return code*(2*math.pi/255)-math.pi

def q8_ste(phase):
    # Forward EXACT q8; derivative = identity (STE) for training
    return phase+(q8(phase)-phase).detach()


assert (len(DISTANCES)==4 and DISTANCES==(0.0241,0.0253,0.0265,0.0277))
print("Fresnel ready:",phys_path)


Team physics SHA256: a98febbb16a9a2e03cbef1f8456dad56cfb92fbec59b4f8049fa6a29db9d7b09
Fresnel ready: d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\src\physics\fresnel.py


## 3. PhysicsSeed + готовая SMP Unet++

In [ ]:
N = 128
WAVELENGTH = 532e-9
PITCH = 10e-6
ind = (torch.arange(N,device=DEVICE,dtype=torch.float32)-N//2)*PITCH
uy,ux = torch.meshgrid(ind,ind,indexing='ij')
IN_CHIRPS = torch.stack([
    torch.exp(1j*math.pi/(WAVELENGTH*z)*(ux.square()+uy.square())) for z in DISTANCES
],dim=0).to(torch.complex64)
coord=torch.linspace(-1,1,128,device=DEVICE,dtype=torch.float32)
ygrid,xgrid=torch.meshgrid(coord,coord,indexing='ij')
XY=torch.stack((xgrid,ygrid),dim=0)[None]

@torch.no_grad()
def physics_seed(targets):
    t=targets.float().to(torch.complex64)
    back_planes=torch.fft.fftshift(
        torch.fft.ifft2(torch.fft.ifftshift(t,dim=(-2,-1)),norm='ortho'),
        dim=(-2,-1)
    )*IN_CHIRPS[None].conj()
    back=back_planes.mean(dim=1)
    seed=torch.stack((back.real,back.imag),dim=1).float()
    maxabs=seed.square().sum(dim=1,keepdim=True).sqrt().amax(dim=(-2,-1),keepdim=True)
    return seed/maxabs.clamp_min(1e-6)

def phase_from_ready_unet(targets):
    seed=physics_seed(targets)
    img_features=(targets.float()-0.45)/0.225
    inp=torch.cat((img_features,seed,XY.expand(len(targets),-1,-1,-1)),dim=1)
    with torch.autocast(device_type=DEVICE.type,enabled=AMP):
        delta=model(inp)
    field=0.7*seed+delta.float()
    return torch.atan2(field[:,1:2],field[:,0:1])

def construct_model(pretrained):
    return smp.UnetPlusPlus(
        encoder_name='resnet34',
        encoder_weights='imagenet' if pretrained else None,
        in_channels=8,classes=2,activation=None
    ).to(DEVICE)

model=None
selected_path=None
checkpoint_info=None
for candidate in CHECKPOINT_CANDIDATES:
    if not candidate.exists():
        continue
    try:
        saved=torch.load(candidate,map_location='cpu',weights_only=False)
        if saved.get('architecture') != 'smp.UnetPlusPlus/resnet34/seed8':
            print('Skipping incompatible checkpoint:',candidate)
            continue
        if model is None: model=construct_model(pretrained=False)
        model.load_state_dict(saved['model'],strict=True)
        checkpoint_info=saved
        selected_path=candidate
        print('LOADED EXISTING WEIGHTS:',selected_path,'epoch=',saved.get('epoch'))
        break
    except (RuntimeError,KeyError,TypeError,ValueError) as exc:
        print('Invalid checkpoint:',candidate,'|',str(exc)[:180])

if selected_path is None:
    try:
        model=construct_model(pretrained=True)
        print('ImageNet encoder pretrained weights loaded.')
    except Exception as exc:
        print('WARNING: ImageNet weights unavailable:',str(exc)[:150])
        print('Retrying with randomly initialized SMP ResNet34 encoder (still training from scratch).')
        model=construct_model(pretrained=False)
    head=model.segmentation_head[0] if isinstance(model.segmentation_head,nn.Sequential) else model.segmentation_head
    if isinstance(head,nn.Conv2d):
        nn.init.normal_(head.weight,mean=0,std=1e-4)
        if head.bias is not None:nn.init.zeros_(head.bias)

assert model is not None
model.eval()
print('SMP Unet++ ready | model params:',sum(p.numel() for p in model.parameters()))


LOADED EXISTING WEIGHTS: d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\checkpoints\best_smp_competition.pt epoch= 6
SMP Unet++ ready | model params: 26094434


## 4. Обучающие данные (только при отсутствии checkpoint)

In [5]:
class RandomQuartets(Dataset):
    def __init__(self,sources,total,seed,augment=True):
        self.sources,self.total,self.seed,self.augment=sources,total,seed,augment
        self.epoch=0
    def __len__(self):return self.total
    def __getitem__(self,index):
        rng=random.Random(self.seed+int(index)*100003+self.epoch*1000000007)
        result=[]
        for _ in range(4):
            ds=rng.choice(self.sources)
            image,_=ds[rng.randrange(len(ds))]
            if not isinstance(image,Image.Image):image=Image.fromarray(np.asarray(image))
            image=image.convert('L')
            if self.augment:
                w,h=image.size
                if rng.random()<0.4 and min(w,h)>16:
                    scale=rng.uniform(.68,1.)
                    nw,nh=max(8,int(w*scale)),max(8,int(h*scale))
                    left,top=rng.randrange(w-nw+1),rng.randrange(h-nh+1)
                    image=image.crop((left,top,left+nw,top+nh))
                if rng.random()<0.5:image=ImageOps.mirror(image)
                if rng.random()<0.5:image=ImageOps.flip(image)
            # Источники >=128: реальные фрагменты, без искусственного растягивания.
            if image.width>=128 and image.height>=128:
                left=rng.randrange(image.width-127) if self.augment else (image.width-128)//2
                top=rng.randrange(image.height-127) if self.augment else (image.height-128)//2
                image=image.crop((left,top,left+128,top+128))
            else:
                image=ImageOps.fit(image,(128,128),method=Image.Resampling.BICUBIC)
            ar=np.asarray(image,dtype=np.float32).copy()/255.
            if self.augment:
                ar=np.clip((ar**rng.uniform(.75,1.45))*rng.uniform(.8,1.2),0,1)
            ar/=max(float(ar.max()),1e-8)
            result.append(torch.from_numpy(ar.astype(np.float32)))
        return torch.stack(result,dim=0)

class LocalImages(Dataset):
    def __init__(self,path):
        self.paths=sorted(p for p in path.rglob('*') if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.bmp','.webp'}) if path.exists() else []
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        with Image.open(self.paths[index]) as im:
            return im.convert('L').copy(),0

def load_sources(is_train):
    sources=[]
    for name,cls in [('CIFAR10',datasets.CIFAR10),('FashionMNIST',datasets.FashionMNIST),('MNIST',datasets.MNIST)]:
        try:
            sources.append(cls(root=str(DATA_ROOT),train=is_train,download=False))
            print('Dataset:',name,'train' if is_train else 'val')
        except (RuntimeError,FileNotFoundError,OSError):
            pass
    hires=LocalImages(DATA_ROOT/'highres'/('train' if is_train else 'val'))
    if len(hires):
        sources.append(hires)
        print('Dataset: highres',len(hires),'train' if is_train else 'val')
    return sources

train_loader=None
val_loader=None
val_ds=None

def get_validation_data():
    global val_loader,val_ds
    if val_ds is None:
        sources=load_sources(is_train=False)
        if not sources:return None
        val_ds=RandomQuartets(sources,VAL_QUARTETS,SEED+100000,augment=False)
        val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=AMP)
    return val_ds


## 5. BEST или обучение с нуля

In [6]:
def pearson_per_plane(p,t):
    a=p-p.mean(dim=(-2,-1),keepdim=True)
    b=t-t.mean(dim=(-2,-1),keepdim=True)
    numerator=(a*b).sum((-2,-1))
    den=(a.square().sum((-2,-1))*b.square().sum((-2,-1))).sqrt()
    return torch.where(den>1e-8,numerator/den.clamp_min(1e-8),torch.zeros_like(den))

def global_ssim_per_plane(p,t):
    mp,mt=p.mean((-2,-1)),t.mean((-2,-1))
    dp,dt=p-mp[...,None,None],t-mt[...,None,None]
    vp,vt=dp.square().mean((-2,-1)),dt.square().mean((-2,-1))
    cov=(dp*dt).mean((-2,-1));c1,c2=.01**2,.03**2
    return (2*mp*mt+c1)*(2*cov+c2)/((mp.square()+mt.square()+c1)*(vp+vt+c2))

def local_ssim_per_plane(p,t,window=7):
    b,c,h,w=p.shape
    a=p.reshape(b*c,1,h,w);q=t.reshape(b*c,1,h,w)
    def avg(x):return F.avg_pool2d(F.pad(x,[window//2]*4,mode='reflect'),window,stride=1)
    ma,mb=avg(a),avg(q)
    va=(avg(a*a)-ma.square()).clamp_min(0)
    vb=(avg(q*q)-mb.square()).clamp_min(0)
    cov=avg(a*q)-ma*mb;c1,c2=.01**2,.03**2
    s=(2*ma*mb+c1)*(2*cov+c2)/((ma.square()+mb.square()+c1)*(va+vb+c2))
    return s.mean((-2,-1)).reshape(b,c)

def custom_loss(pred,target):
    cc=pearson_per_plane(pred,target)
    loc=local_ssim_per_plane(pred,target)
    glob=global_ssim_per_plane(pred,target)
    mse=(pred-target).square().mean((-2,-1))
    plane=.50*(1-loc)+.27*(1-cc)+.18*(1-glob)+.05*mse
    return plane.mean()+.20*plane.amax(dim=1).mean(),{'cc':cc,'local':loc,'global':glob}

def calc_metrics(pred,target):
    _,m=custom_loss(pred,target)
    return {key+'_mean':float(x.mean()) for key,x in m.items()} | {
        key+'_planes':[float(v) for v in x.mean(0)] for key,x in m.items()
    }

def save_atomic(obj,path):
    temporary=path.with_name(path.name+'.tmp')
    torch.save(obj,temporary)
    os.replace(temporary,path)

@torch.no_grad()
def validate(loader):
    model.eval()
    accum=np.zeros(4,dtype=np.float64); count=0
    for targets in loader:
        targets=targets.to(DEVICE,non_blocking=True)
        phase=phase_from_ready_unet(targets)
        rec=norm_amp(reconstruct_four(q8(phase.float())))
        loss,m=custom_loss(rec,targets)
        n=len(targets);count+=n
        accum+=n*np.array([float(loss),float(m['cc'].mean()),
                           float(m['local'].mean()),float(m['global'].mean())])
    val,cc,loc,glob=(accum/count).tolist()
    return {'val_loss':val,'cc':cc,'local':loc,'global':glob,
            'score_internal':.30*cc+.40*loc+.30*glob}

if selected_path is None:
    train_sources=load_sources(is_train=True)
    val_sources=load_sources(is_train=False)
    if not train_sources or not val_sources:
        raise FileNotFoundError('No compatible BEST. For training, put CIFAR10/MNIST/FashionMNIST in ./data or images in data/highres/train and data/highres/val.')
    train_ds=RandomQuartets(train_sources,TRAIN_QUARTETS_PER_EPOCH,SEED,augment=True)
    train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=AMP)
    get_validation_data()
    optimizer=torch.optim.AdamW([
        {'params':model.encoder.parameters(),'lr':LR*.3},
        {'params':list(model.decoder.parameters())+list(model.segmentation_head.parameters()),'lr':LR},
    ],weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=TRAIN_EPOCHS_IF_NO_CHECKPOINT,eta_min=LR*.06)
    scaler=torch.amp.GradScaler('cuda',enabled=AMP)
    best_score=-float('inf');start_epoch=0;misses=0
    if FINAL_LAST.exists():
        try:
            resume=torch.load(FINAL_LAST,map_location='cpu',weights_only=False)
            assert resume.get('architecture')=='smp.UnetPlusPlus/resnet34/seed8'
            model.load_state_dict(resume['model'],strict=True)
            optimizer.load_state_dict(resume['optimizer'])
            scheduler.load_state_dict(resume['scheduler'])
            scaler.load_state_dict(resume['scaler'])
            start_epoch=int(resume['epoch'])
            best_score=float(resume['best_score'])
            misses=int(resume.get('misses',0))
            print('Resuming interrupted training at epoch:',start_epoch)
        except Exception as exc:
            print('Cannot resume LAST, training from scratch:',str(exc)[:180])
            # For invalid LAST, restore original model/optimizer state by removing incompatible LAST manually.
            raise RuntimeError('Invalid FINAL_LAST. Rename/remove it before training from scratch.') from exc
    begin=time.monotonic()
    for epoch in range(start_epoch,TRAIN_EPOCHS_IF_NO_CHECKPOINT):
        if (time.monotonic()-begin)/3600>MAX_TRAIN_HOURS:
            print('Reached training time cap after completed epochs.');break
        train_ds.epoch=epoch
        model.train();seen=0;loss_total=0.
        quantize=epoch>=12
        bar=tqdm(train_loader,desc=f'Train {epoch+1}/{TRAIN_EPOCHS_IF_NO_CHECKPOINT}',unit='batch',dynamic_ncols=True)
        for targets in bar:
            targets=targets.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            phase=phase_from_ready_unet(targets)
            rec=norm_amp(reconstruct_four(q8_ste(phase.float()) if quantize else phase.float()))
            loss,_=custom_loss(rec,targets)
            if not torch.isfinite(loss):raise FloatingPointError('Nonfinite training loss')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            scaler.step(optimizer);scaler.update()
            seen+=len(targets);loss_total+=float(loss.detach())*len(targets)
            bar.set_postfix(loss=f'{loss_total/seen:.4f}',best=f'{best_score:.4f}')
        scheduler.step()
        metrics=validate(val_loader)
        score=metrics['score_internal']
        improved=score>best_score+1e-5
        if improved:
            best_score=score; misses=0
            save_atomic({'model':model.state_dict(),
                         'architecture':'smp.UnetPlusPlus/resnet34/seed8',
                         'epoch':epoch+1,'metrics':metrics},FINAL_BEST)
        else:misses+=1
        save_atomic({'model':model.state_dict(),'optimizer':optimizer.state_dict(),
                     'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
                     'epoch':epoch+1,'best_score':best_score,'misses':misses,
                     'architecture':'smp.UnetPlusPlus/resnet34/seed8'},FINAL_LAST)
        print(f'epoch {epoch+1:02d}: train={loss_total/seen:.4f} '
              f'CC={metrics["cc"]:.4f} local={metrics["local"]:.4f} '
              f'global={metrics["global"]:.4f} BEST={best_score:.4f} '
              f'Q8={quantize}',flush=True)
        if epoch+1>=MIN_EPOCHS and misses>=PATIENCE:
            print('Early stopping.');break
    if not FINAL_BEST.exists():
        raise RuntimeError('Training stopped before first saved BEST. Increase MAX_TRAIN_HOURS.')
    selected_path=FINAL_BEST
    checkpoint_info=torch.load(selected_path,map_location='cpu',weights_only=False)
    model.load_state_dict(checkpoint_info['model'],strict=True)
    print('Training complete. Selected BEST:',selected_path)
else:
    print('Using existing trained model. No new training or benchmark.')
model.eval()


Using existing trained model. No new training or benchmark.


UnetPlusPlus(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(8, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, ep

## 6. Генерация одного ДОЭ и четырёх восстановлений

In [7]:
def read_four_inputs(paths):
    if len(paths)!=4:raise ValueError('Exactly four input PNGs required')
    tensors=[]
    for path in paths:
        with Image.open(path) as im:
            if im.size!=(128,128):raise ValueError(f'{path}: expected 128x128, got {im.size}')
            if im.mode!='L':raise ValueError(f'{path}: expected grayscale L PNG, got {im.mode}')
            arr=np.asarray(im,dtype=np.float32).copy()/255.
        tensors.append(torch.from_numpy(arr))
    result=torch.stack(tensors)[None].to(DEVICE)
    return result/result.amax((-2,-1),keepdim=True).clamp_min(1e-8)

def predict_and_export(targets,out_dir,source):
    out_dir=Path(out_dir);out_dir.mkdir(parents=True,exist_ok=True)
    targets=targets.to(DEVICE)
    if targets.shape!=(1,4,128,128):raise ValueError(f'Input must be (1,4,128,128), got {targets.shape}')
    model.eval()
    if AMP:torch.cuda.synchronize()
    start=time.perf_counter()
    with torch.inference_mode():phase=phase_from_ready_unet(targets)
    if AMP:torch.cuda.synchronize()
    generation_seconds=time.perf_counter()-start
    phase_to_png(phase[0,0],out_dir/'doe.png')
    with Image.open(out_dir/'doe.png') as im:
        assert im.mode=='L' and im.size==(128,128) and np.asarray(im).dtype==np.uint8
    loaded=phase_from_png(out_dir/'doe.png')
    with torch.no_grad():
        recon=norm_amp(reconstruct_four(loaded))
        metrics=calc_metrics(recon,targets)
    assert torch.isfinite(recon).all(), 'Nonfinite Fresnel reconstruction'
    for k in range(4):
        for name,arr in [('reconstructed',recon[0,k]),('target',targets[0,k])]:
            pixels=(arr.detach().cpu().numpy()*255).round().clip(0,255).astype(np.uint8)
            Image.fromarray(pixels,mode='L').save(out_dir/f'{name}_{k+1}.png')
    info={'input_source':source,'checkpoint':str(selected_path.resolve()),
          'model':'smp.UnetPlusPlus/resnet34/seed8','phase_generation_seconds':generation_seconds,
          'metrics_internal_not_official':metrics,
          'physics_sha256':hashlib.sha256(phys_path.read_bytes()).hexdigest(),
          'iterative_refinement_steps':0}
    (out_dir/'result.json').write_text(json.dumps(info,indent=2,ensure_ascii=False),encoding='utf-8')
    return info

if all(p.is_file() for p in INPUT_FILES):
    targets=read_four_inputs(INPUT_FILES)
    source='four supplied grayscale PNGs'
elif not any(p.is_file() for p in INPUT_FILES):
    demo=get_validation_data()
    if demo is None:
        raise FileNotFoundError('Put four grayscale 128x128 PNGs at '+str(INPUT_FILES)+'; no local test dataset for demonstration.')
    targets=demo[0][None].to(DEVICE)
    source='DEMO: public dataset test split, not organizer images'
else:
    raise FileNotFoundError('Incomplete contest input: supply all four input_1.png ... input_4.png.')

result=predict_and_export(targets,OUT,source)
READY_MARKER.write_text(json.dumps({
    'status':'complete','checkpoint':str(selected_path.resolve()),
    'doe':str((OUT/'doe.png').resolve()),
    'outputs':str(OUT.resolve()),
    'notebook':'Nuclear_Competition_Final.ipynb',
    'iteration_based_phase_optimization':False,
},indent=2,ensure_ascii=False),encoding='utf-8')
print('DONE. Model checkpoint:',selected_path)
print('DOE:',OUT/'doe.png')
print('4 reconstructions:',*[str(OUT/f'reconstructed_{k}.png') for k in range(1,5)])
print('CC:',result['metrics_internal_not_official']['cc_mean'],
      'Local SSIM:',result['metrics_internal_not_official']['local_mean'],
      'Global SSIM:',result['metrics_internal_not_official']['global_mean'])
print('Codex marker:',READY_MARKER)


Dataset: CIFAR10 val
Dataset: FashionMNIST val
Dataset: MNIST val
DONE. Model checkpoint: d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\checkpoints\best_smp_competition.pt
DOE: d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\outputs\final_submission\doe.png
4 reconstructions: d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\outputs\final_submission\reconstructed_1.png d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\outputs\final_submission\reconstructed_2.png d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\outputs\final_submission\reconstructed_3.png d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\outputs\final_submission\reconstructed_4.png
CC: 0.49198323488235474 Local SSIM: 0.34244880080223083 Global SSIM: 0.3658555746078491
Codex marker: d:\ML.Andan\Year_2026-2027\hackathons\NuclearITHack\CODEX_READY.json


## 7. Для Codex

Перенести `physics_seed`, `phase_from_ready_unet`, `read_four_inputs`, `phase_to_png` и `predict_and_export` в `submission/src/`, приложить выбранный checkpoint, сделать CPU-совместимый Docker/CLI. В Docker **не должно быть** обучения, скачивания весов, датасетов и оптимизации фазы. Вход — четыре PNG 128×128, выход — `doe.png`.